# Build Constructors Dimension
1. Read silver `constructors` table
2. read gold `ref_nationality_region` table
3. Join the data from `constructors` with `ref_nationality_region` using `nationality`
4. select the required columns
    - constructors.constructor_id
    - constructors.constrcutor_name
    - constructors.nationality
    - ref_nationality_region.region
5. Write the transformed data to gold `dim_constructors` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
silver_constructors = f'{catalog_name}.{silver_schema}.constructors'
nationality_ref =f'{catalog_name}.{gold_schema}.ref_nationality_region'
target_table = f'{catalog_name}.{gold_schema}.dim_constructors'

#### Step 1 - Read source tables
- `silver.constructors`
- `gold.ref_nationality_region`

In [0]:
constructor_df = spark.table(silver_constructors)
ref_nationality_region_df = spark.table(nationality_ref)

#### Step 2 - Join `constructors` with `nationality_region_df` using `nationality`

Select the following columns

1. constructors.constructor_id
2. constructors.constructor_name
3. constructors.nationality
4. ref_nationality_region.region

In [0]:
dim_constructors = (
    constructor_df
        .join(
            ref_nationality_region_df,
            constructor_df.nationality == ref_nationality_region_df.nationality,
            'left'
        )
        .select(
            constructor_df.constructor_id,
            constructor_df.constructor_name,
            constructor_df.nationality,
            ref_nationality_region_df.region.alias("nationality_region")
        )
)

#### Step 3 - Write the transformed data to the `gold` `dim_constructors` table

In [0]:
(
    dim_constructors
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
display(spark.table(target_table))